In [ ]:
#setup
!pip install telethon==1.36.0 confluent-kafka
!pip install --upgrade confluent-kafka

In [1]:
#setup a local sql table to save messages
import sqlite3

def init_db(db_path="messages.db"):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            channel TEXT,
            sender TEXT,
            text TEXT,
            ts INTEGER,
            PRIMARY KEY (channel, ts, sender, text)
        )
    """)
    conn.commit()
    conn.close()
    print("Database ready: messages.db")

init_db()

Database ready: messages.db


In [ ]:
import json
import sqlite3
import asyncio
from telethon import TelegramClient
from telethon.errors import FloodWaitError

with open("config.json", "r") as f:
    config = json.load(f)

API_ID = config["api_id"]
API_HASH = config["api_hash"]
SESSION_NAME = config.get("session_name", "scraper_session")
CHANNELS = config["channels"]

async def scrape_history(limit_per_channel, delay_between_channels):
    client = TelegramClient(SESSION_NAME, API_ID, API_HASH)
    await client.start()
    
    print("Loading joined dialogs...")
    dialogs = await client.get_dialogs()
    dialog_map = {d.title: d.entity for d in dialogs}
    
    conn = sqlite3.connect("messages.db")
    cur = conn.cursor()
    
    for ch in CHANNELS:
        entity = dialog_map.get(ch)
        if not entity:
            try:
                entity = await client.get_entity(ch)
            except Exception as e:
                print(f"Could not resolve '{ch}': {e}")
                continue
                
        channel_label = getattr(entity, "title", str(ch))
        print(f"Scraping '{channel_label}'...")
        count = 0
        
        try:
            async for msg in client.iter_messages(entity, limit=limit_per_channel):
                if not msg.text:
                    continue
                
                sender = str(msg.sender_id or "unknown")
                ts = int(msg.date.timestamp())
                
                cur.execute("""
                    INSERT OR IGNORE INTO messages (channel, sender, text, ts)
                    VALUES (?, ?, ?, ?)
                """, (channel_label, sender, msg.text, ts))
                count += 1
                
        except FloodWaitError as e:
            print(f"Hit Telegram rate limit. Sleeping for {e.seconds} seconds...")
            await asyncio.sleep(e.seconds)
            
        conn.commit()
        print(f"Saved {count} messages from '{channel_label}'. Waiting {delay_between_channels}s...")
        await asyncio.sleep(delay_between_channels)
        
    conn.close()
    await client.disconnect()
    print("All channels scraped successfully!")

await scrape_history(limit_per_channel=10000, delay_between_channels=2.0)

Scraping https://t.me/koahadasotbatelegram...
Saved 2989 messages from https://t.me/koahadasotbatelegram
Scraping https://t.me/newsonlineils...
Saved 2884 messages from https://t.me/newsonlineils
Scraping https://t.me/newsonlineils...
Saved 2884 messages from https://t.me/newsonlineils
Scraping https://t.me/danielamram3...
Saved 2827 messages from https://t.me/danielamram3
Scraping complete!


In [4]:
conn = sqlite3.connect("messages.db")
cur = conn.cursor()

cur.execute("SELECT channel, count(*) FROM messages GROUP BY channel")
for row in cur.fetchall():
    print(f"Channel '{row[0]}': {row[1]} messages")

conn.close()

Channel 'https://t.me/danielamram3': 2827 messages
Channel 'https://t.me/koahadasotbatelegram': 2989 messages
Channel 'https://t.me/newsonlineils': 2883 messages
